# Pipeline to streamline the following tasks

 1. Extract YT video urls to create a curated dataset  
 >[VIDEO_ID, VIDEO_TITLE, VIDEO_URL, Duration, Channel]  

 2. Extract audios from the videos using URLS or video_IDs.
 >Strip the .mp3 codec audio files from the videos and save them as a dataset.
 >> DataSet > VideoID > [audio.mp3]

 [3]. Establish process to extract / generate transcriptions from the videos.
 >Likely use of AI models for transcription due to no presence of youtube generated transcripts in the videos.

In [ ]:
# Mount Drive

from google.colab import drive
drive.mount('/content/drive')

In [1]:
#Module imports

!pip install youtube_transcript_api
!pip install --upgrade transformers
!apt-get update
!apt-get install -y ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 69.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu

In [ ]:
#READ DF

df = pd.read_csv('/content/drive/MyDrive/AnnamAI Tasks/YT_DATASET.csv')
finished_indexes = []


### Attempt to source Transcripts from YT id available

In [3]:
from youtube_transcript_api import YouTubeTranscriptApi

def fetch_YT_transcripts(video_id, finished_index):
    try:
        api = YouTubeTranscriptApi()
        # fetch() returns a FetchedTranscriptDocstring object
        raw_transcript = api.fetch(video_id, languages=['en'])

        finished_indexes.append(True)

        # FIX: Use dot notation properties instead of dictionary keys
        for entry in raw_transcript:
            print(f"[{entry.start}s]: {entry.text}")

    except Exception as e:
        finished_index[-1] = False
        print(f"Could not retrieve transcript: {e}")

In [ ]:

for id in YT_VIDEO_ID:
    fetch

## Whisper implementation for transcription with initial prompts



In [1]:
## Whisper implementation for transcription with initial prompts

import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

initial_prompt = "You are a strict transcription model. Process the audio to output the most accurate transcripts. MAKE SURE to keep quantitative words as words. DO NOT CONVERT QUANTITATIVE WORDS INTO NUMBERS/DIGITS"

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    dtype=torch_dtype,
    device=device,
)


config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.71M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

In [3]:
# import torch
# from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

# device = "cuda:0" if torch.cuda.is_available() else "cpu"
# torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# model_id = "openai/whisper-large-v3"

# # 1. Load Model
# model = AutoModelForSpeechSeq2Seq.from_pretrained(
#     model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
# )
# model.to(device)

# # 2. Load Processor
# processor = AutoProcessor.from_pretrained(model_id)

# # 3. Define the pipeline setup
# pipe = pipeline(
#     "automatic-speech-recognition",
#     model=model,
#     tokenizer=processor.tokenizer,
#     feature_extractor=processor.feature_extractor,
#     torch_dtype=torch_dtype,
#     device=device,
# )

# --- INFERENCE SETUP ---

audio_file_path = "audio.mp3"  # Replace with your actual audio file path
initial_prompt = "आप एक सख्त प्रतिलेखन (transcription) मॉडल हैं। सबसे सटीक प्रतिलेखन आउटपुट करने के लिए ऑडियो को प्रोसेस करें। सुनिश्चित करें कि मात्रात्मक शब्दों (संख्याओं) को शब्दों में ही रखें। संख्याओं को अंकों में न बदलें।"
# 4. Tokenize the initial prompt explicitly into integer IDs
# [1:-1] strips out standard bos/eos boundary tokens so it acts cleanly as a prefix
prompt_ids = torch.tensor(processor.tokenizer(initial_prompt, add_special_tokens=False)["input_ids"]).to(device)

# 5. Execute the pipeline passing the token array inside generate_kwargs
result = pipe(
    audio_file_path,
    return_timestamps = True,
    language="hi"
)

print(result["text"])

Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


 नमस्कार, क्रिशेदर्शन कारिकरम में आप सभी दर्शकों का स्वागत है। विश्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च्च् इसमें चने में जो फलियाओं उनकी फलियों में दाना स्वस्थ हो और हमें उत्पा��������������������������������������������������������������������������������������������������������������������������������������������������������������������� आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको आपको इसकी रोगधाम के लिए किसानों को पहले सही तयार रहना चाहिए। सबसे पहले इसकी निगरानी के लिए हम प्रकाश प्रपंच नमक यंद्र का उप्योग करते हैं। एक हेक्टेर में हम इसको अल������������������������������������������������������������������������ कि कितनी हानी हो सकती है हमाई फसल को पतंगों की संख्या गिनकर और इसके अलावा जगा पर हमें लग�������������������������������